# 01 — 実行環境とコード地図

どのコードが正本で、どこから実行され、どの機能が標準経路で有効かを固定します。

**前提**: `00_curriculum_map.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
# 背景: 各章を同じ作業ディレクトリと依存関係で再実行するには、リポジトリの基準パスと外部実装の場所を最初に固定する必要がある。
# 目的: Quadruped-PyMPCをimport可能にし、acadosとヘッドレスMuJoCoの実行環境を後続セルへ引き渡す。
# ファイルシステム上の基準パスを型安全に扱うためPathを読み込む。
from pathlib import Path
# 環境変数の設定とPythonのモジュール探索パス更新に必要な標準ライブラリを読み込む。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化し、教材全体の基準候補とする。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけ、リポジトリ直下へ基準を1階層戻す。
if ROOT.name == "notebook_pympc":
    # externalディレクトリを参照できるリポジトリ直下へROOTを合わせる。
    ROOT = ROOT.parent
# 上流制御実装が置かれたQuadruped-PyMPCの絶対パスを構成する。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動位置のまま進まず、依存リポジトリの欠落を具体的なパス付きで検出する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同じパスを重複登録せず、まだimport探索対象でない場合だけ追加する。
if str(PYMPC_ROOT) not in sys.path:
    # ローカルのquadruped_pympcパッケージを通常のimport文で読めるよう探索順の先頭へ置く。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物・共有資源の基準位置を未設定時だけ登録し、利用者の明示設定は上書きしない。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のない環境でもMuJoCoを描画可能にするため、未設定時のOpenGL backendをEGLにする。
os.environ.setdefault("MUJOCO_GL", "egl")
# 実際に採用されたworkspace基準を表示し、相対パス問題を診断できるようにする。
print("workspace :", ROOT)
# import対象となる上流実装の場所を表示し、参照しているコード版を確認可能にする。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 起動方法

リポジトリ直下で次を実行してからJupyterを開きます。

`source .env.workshop && uv run --extra workshop jupyter lab`

`.env.workshop` は acados の共有ライブラリとヘッドレスMuJoCoを設定します。
Notebook単独の `os.environ` では、既に起動済みプロセスの共有ライブラリ探索を
完全には変更できないため、シェルでの `source` が確実です。

In [2]:
# 背景: 数値計算・物理シミュレーション・記号モデル・OCP生成の依存パッケージが欠けると、後続章は異なる地点で失敗する。
# 目的: 必須パッケージを動的にimportし、利用可否と取得可能な版番号を一括表示して環境を早期診断する。
# パッケージ名の文字列からモジュールを読み込むため標準のimportlibを使う。
import importlib
# 後続章が直接または上流実装経由で必要とする依存パッケージ名を列挙する。
packages = ["numpy", "scipy", "mujoco", "casadi", "gym_quadruped", "acados_template"]
# 各依存を個別に検査し、1件の失敗で残りの診断を中断しないよう順番に処理する。
for name in packages:
    # import時例外をパッケージ単位で捕捉するため保護区間を開始する。
    try:
        # 現在のPython環境から対象名のモジュールを読み込み、成功時は実体を保持する。
        module = importlib.import_module(name)
        # __version__がないパッケージも空欄として扱い、固定幅の成功一覧を表示する。
        print(f"{name:18s} OK  {getattr(module, '__version__', '')}")
    # 欠落だけでなく共有ライブラリ不整合など全import例外を診断情報として受け取る。
    except Exception as exc:
        # 失敗した名前・例外型・内容を表示し、環境修復の手掛かりを残す。
        print(f"{name:18s} NG  {type(exc).__name__}: {exc}")

numpy              OK  2.4.6
scipy              OK  1.17.1
mujoco             OK  3.11.0
casadi             OK  3.7.2
gym_quadruped      OK  


acados_template    OK  


## 標準経路の正本

- 入口: `simulation/simulation.py::run_simulation`
- 統合: `quadruped_pympc_wrapper.py::compute_actions`
- 参照生成: `interfaces/wb_interface.py`
- MPC選択: `interfaces/srbd_controller_interface.py`
- 標準力学: `controllers/gradient/nominal/centroidal_model_nominal.py`
- 標準OCP: `controllers/gradient/nominal/centroidal_nmpc_nominal.py`
- 設定: `quadruped_pympc/config.py`

`sampling`、`input_rates`、`kinodynamic` などは分岐先です。まず `type='nominal'` を
一周してから比較します。

In [3]:
# 背景: 実験結果を解釈するには、robot・controller・時間刻み・gait・sceneという基準条件をコードの正本から固定する必要がある。
# 目的: 現行configからbaseline条件を辞書へ集約し、後続章で使う値と単位を一目で確認する。
# 上流実装が実際に参照する設定値を同じ場所から取得するためconfigを読み込む。
from quadruped_pympc import config as cfg
# 再現性に影響する主要設定を、意味の分かるキーで1つの辞書へまとめる。
baseline = {
    "robot": cfg.robot,  # MuJoCoと制御設定が対象とする四足robot名を保存する。
    "controller": cfg.mpc_params["type"],  # 使用するSRBD controller分岐名を保存する。
    "N": cfg.mpc_params["horizon"],  # MPCの予測段数N[-]を保存する。
    "dt_mpc": cfg.mpc_params["dt"],  # 予測モデル1段の時間刻みdt_mpc[s]を保存する。
    "dt_sim": cfg.simulation_params["dt"],  # MuJoCo積分1 stepの刻みdt_sim[s]を保存する。
    "mpc_frequency": cfg.simulation_params["mpc_frequency"],  # MPC呼出し設定[Hz]を保存する。
    "gait": cfg.simulation_params["gait"],  # 位相offsetとduty factorを選ぶgait名を保存する。
    "scene": cfg.simulation_params["scene"],  # 接触条件を決めるMuJoCo scene名を保存する。
}
# Notebookの最終式として辞書を評価し、Jupyterの整形表示でbaseline全体を確認する。
baseline

{'robot': 'go2',
 'controller': 'nominal',
 'N': 12,
 'dt_mpc': 0.02,
 'dt_sim': 0.002,
 'mpc_frequency': 100,
 'gait': 'trot',
 'scene': 'perlin'}

`mpc_frequency=100` はHzではなく、この実装では
`step_num % round(1/(mpc_frequency*simulation_dt))` に使われます。
`dt_sim=0.002` なら5シミュレーションstepごと、すなわち100 Hzです。
名前だけで意味を推測せず、使用箇所まで追う習慣をつけます。

## 標準nominal経路を10ブロックで読む

以下は`simulation.py::run_simulation`から`env.step`までの実装順です。後続Notebook 02–12が各ブロックを実行して分解します。

1. **Plant観測** (`simulation/simulation.py`): \(y=(p,v,R,\omega,p_i,J_i,M_i,q,\dot q)\)
2. **指令** (`target_base_vel`): \(r=(v^{ref},\omega^{ref})\)
3. **Gait** (`PeriodicGaitGenerator`): \(\phi_i^+=(\phi_i+f\Delta t)\bmod1\), \(c_i=1[\phi_i<\beta]\)
4. **Foothold** (`FootholdReferenceGenerator`): \(p_i^{ref}=p_{hip,i}+T_{st}v^{ref}/2+\sqrt{h/g}(v-v^{ref})\)
5. **MPC状態・参照** (`WBInterface`): dictを状態ベクトル \(x\) と参照 \(x^{ref}\) へ対応づける
6. **SRBD** (`Centroidal_Model_Nominal`): \(m\dot v=\sum c_iF_i+mg\), \(I\dot\omega+\omega\times I\omega=\sum(r_i-p)\times c_iF_i\)
7. **OCP** (`Acados_NMPC_Nominal`): \(\min\sum\|x-x^{ref}\|_Q^2+\|u-u^{ref}\|_R^2\), 摩擦・接触制約付き
8. **Receding horizon** (`SRBDControllerInterface`): 第0段のGRFとfootholdだけを現在の制御へ渡す
9. **Torque** (`WBInterface`): stanceは \(\tau=-J^TF\)、swingはCartesian PD + feedback linearization
10. **Plant更新** (`simulation.py`): actuator順へ詰め、soft clip後に`env.step(action)`

次のコードは上流の呼出し関係を短くした**読解用の同型コード**です。省略した引数は`...`で示し、各コメントの式番号を上のブロック番号と対応させます。

In [4]:
# 背景: nominal制御周期は複数ファイルに分散しており、観測からトルク適用までの呼出し順と式番号を同時に追うための短い地図が必要である。
# 目的: 上流の10ブロックを実行しない擬似コードとして保存し、各信号のshape・単位・座標系と後続Notebookの対応を示す。
# 読解用地図を再利用可能な関数として定義し、実シミュレーションを起動せず説明文字列だけを返す。
def nominal_control_cycle_reading_map():
    """実行用ではなく、上流の1周期を式と対応させた読解用コード。"""

    # [1] Plant観測 y: worldの並進量、baseの角速度、全身力学量を読む
    # feet_pos = env.feet_pos(frame="world")             # [m], LegsAttr(4×3)
    # base_lin_vel = env.base_lin_vel(frame="world")     # [m/s], (3,)
    # base_ang_vel = env.base_ang_vel(frame="base")      # [rad/s], (3,)
    # J = env.feet_jacobians(frame="world")              # 足速度 = J qdot

    # [2] 参照 r。heading指令をworldへ回した値が返る
    # ref_lin_w, ref_ang_w = env.target_base_vel()

    # [3–5] WBInterfaceが位相→接地列、着地点、MPC用state/refを一括生成
    # state, ref, contact_seq, step_h, optimize_swing = wb.update_state_and_reference(
    #     com_pos, base_pos, base_lin_vel, rpy, base_ang_vel,
    #     feet_pos, hip_pos, joints_pos, heightmaps, legs, dt_sim,
    #     ref_lin_w, ref_ang_w, mujoco_contact,
    # )

    # [6–8] 5 sim stepごとにSRBD OCPを解く。
    # u* = [foot_velocity(12), GRF(12)]、現在使うのは第0予測段。
    # if step_num % round(1 / (mpc_frequency * dt_sim)) == 0:
    #     grf, foothold, *_ = srbd.compute_control(
    #         state, ref, contact_seq, inertia, phase, step_freq, optimize_swing
    #     )

    # [9] 接地脚は tau=-J.T@F、遊脚はspline参照のCartesian PDを使う
    # tau, q_des, qd_des = wb.compute_stance_and_swing_torque(
    #     dt_sim, q, qdot, J, Jdot, feet_pos, feet_vel, passive, bias,
    #     mass_matrix, grf, foothold, q_idx, v_idx, previous_tau, ...
    # )

    # [10] 12 actuator順へ格納し、物理上限の90%でclipしてPlantを1 step進める
    # action[env.legs_tau_idx.FL] = np.clip(tau.FL, tau_min.FL, tau_max.FL)
    # state_next, reward, terminated, truncated, info = env.step(action)
    # 実行可能な各ブロックの参照先を案内する文字列を呼出し元へ返す。
    return "See notebooks 02–12 for executable versions of blocks 1–10."

# 読解用関数を1回呼び、実コードは後続章にあることをNotebook出力へ表示する。
print(nominal_control_cycle_reading_map())

See notebooks 02–12 for executable versions of blocks 1–10.


## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。